In [1]:
import datetime
import pandas as pd

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *

[2024-05-14 18:16:48,982][INFO]: Connecting jbhagat@aeon-db2:3306
[2024-05-14 18:16:49,005][INFO]: Connected jbhagat@aeon-db2:3306
[2024-05-14 18:16:49,005][INFO]: Connected jbhagat@aeon-db2:3306


In [2]:
dj.conn()

DataJoint connection (connected) jbhagat@aeon-db2:3306

In [3]:
# get key from table with multiple subjects
#key = (BlockAnalysis.aggr(BlockAnalysis.Subject, subj_count="count(*)") & "subj_count>1").fetch("KEY")[-1]
key = {"experiment_name": "social0.2-aeon3", "block_start": "2024-02-11 13:36:10"}
print(key)

{'experiment_name': 'social0.2-aeon3', 'block_start': '2024-02-11 13:36:10'}


## Compute per-subject patch preference, by wheel distance spun, and time on patch

In [ ]:
"""Patch Preference in Block"""

# Get all patches

# For each patch
#     For each subject
#         Get patch preference over duration of block
#         Get patch preference from halfway through block to end (as found by `n_pellets // 2`)

In [7]:
block_patches = (BlockAnalysis.Patch & key).fetch(as_dict=True)
block_subjects = (BlockAnalysis.Subject & key).fetch(as_dict=True)
subject_names = [s["subject_name"] for s in block_subjects]
patch_names = [p["patch_name"] for p in block_patches]
# Construct subject position dataframe
subjects_positions_df = pd.concat(
    [
        pd.DataFrame(
            {"subject_name": [s["subject_name"]] * len(s["position_timestamps"])}
            | {
                k: s[k]
                for k in (
                    "position_timestamps",
                    "position_x",
                    "position_y",
                    "position_likelihood",
                )
            }
        )
        for s in block_subjects
    ]
)
subjects_positions_df.set_index("position_timestamps", inplace=True)

### Computations

In [5]:
in_patch_radius = 130  # pixels
pref_attrs = ["cum_dist", "cum_time", "cum_pref_dist", "cum_pref_time"]
cols = [s + "_" + p + "_" + a for s in subject_names for p in patch_names for a in pref_attrs]
all_subj_patch_pref = pd.DataFrame(columns=cols)
for patch in block_patches:
    cum_wheel_dist = pd.Series(index=patch["wheel_timestamps"], data=patch["wheel_cumsum_distance_travelled"])
    # Get patch x,y from metadata patch rfid loc
    patch_center = (
        streams.RfidReader * streams.RfidReader.Attribute
        & key
        & f"rfid_reader_name LIKE '%{patch['patch_name']}%'"
        & "attribute_name = 'Location'"
    ).fetch1("attribute_value")
    patch_center = (int(patch_center["X"]), int(patch_center["Y"]))
    subjects_xy = subjects_positions_df[["position_x", "position_y"]].values
    dist_to_patch = np.sqrt(np.sum((subjects_xy - patch_center) ** 2, axis=1).astype(float))
    dist_to_patch_df = subjects_positions_df[["subject_name"]].copy()
    dist_to_patch_df["dist_to_patch"] = dist_to_patch
    # Assign pellets and wheel timestamps to subjects
    if len(block_subjects) == 1:
        cum_wheel_dist_dm = cum_wheel_dist.to_frame(name=subject_names[0])
        patch_df_for_pellets_df = pd.DataFrame(
            index=patch["pellet_timestamps"], data={"subject_name": subject_names[0]}
        )
    else:
        # Assign id based on which subject was closest to patch at time of event
        # Get distance-to-patch at each wheel ts and pel del ts, organized by subject
        dist_to_patch_wheel_ts_id_df = pd.DataFrame(index=cum_wheel_dist.index, columns=subject_names)
        dist_to_patch_pel_ts_id_df = pd.DataFrame(index=patch["pellet_timestamps"], columns=subject_names)
        for subject_name in subject_names:
            # Find closest match between pose_df indices and wheel indices
            if not dist_to_patch_wheel_ts_id_df.empty:
                dist_to_patch_wheel_ts_subj = pd.merge_asof(
                    left=pd.DataFrame(dist_to_patch_wheel_ts_id_df[subject_name].copy()).reset_index(
                        names="time"
                    ),
                    right=dist_to_patch_df[
                        dist_to_patch_df["subject_name"] == subject_name
                    ].copy().reset_index(names="time"),
                    on="time",
                    # left_index=True,
                    # right_index=True,
                    direction="nearest",
                    tolerance=pd.Timedelta("100ms"),
                )
                dist_to_patch_wheel_ts_id_df[subject_name] = dist_to_patch_wheel_ts_subj["dist_to_patch"].values
            # Find closest match between pose_df indices and pel indices
            if not dist_to_patch_pel_ts_id_df.empty:
                dist_to_patch_pel_ts_subj = pd.merge_asof(
                    left=pd.DataFrame(dist_to_patch_pel_ts_id_df[subject_name].copy()).reset_index(
                        names="time"
                    ),
                    right=dist_to_patch_df[
                        dist_to_patch_df["subject_name"] == subject_name
                    ].copy().reset_index(names="time"),
                    on="time",
                    # left_index=True,
                    # right_index=True,
                    direction="nearest",
                    tolerance=pd.Timedelta("200ms"),
                )
                dist_to_patch_pel_ts_id_df[subject_name] = dist_to_patch_pel_ts_subj["dist_to_patch"].values
        # Get closest subject to patch at each pel del timestep
        patch_df_for_pellets_df = pd.DataFrame(
            index=patch["pellet_timestamps"],
            data={"subject_name": dist_to_patch_pel_ts_id_df.idxmin(axis=1).values},
        )
        # Get closest subject to patch at each wheel timestep
        cum_wheel_dist_subj_df = pd.DataFrame(index=cum_wheel_dist.index, columns=subject_names, data=0.0)
        closest_subjects = dist_to_patch_wheel_ts_id_df.idxmin(axis=1)
        wheel_dist = cum_wheel_dist.diff().fillna(cum_wheel_dist.iloc[0])
        # Assign wheel dist to closest subject for each wheel timestep
        for subject_name in subject_names:
            subj_idxs = cum_wheel_dist_subj_df[closest_subjects == subject_name].index
            cum_wheel_dist_subj_df.loc[subj_idxs, subject_name] = wheel_dist[subj_idxs]
        cum_wheel_dist_subj_df = cum_wheel_dist_subj_df.cumsum(axis=0)
        # In patch time
        in_patch = dist_to_patch_wheel_ts_id_df < in_patch_radius
        # Fill in `all_subj_patch_pref`
        for subject_name in subject_names:
            all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_dist"] = (
                cum_wheel_dist_subj_df[subject_name].values
            )
            dt = (in_patch.index[1] - in_patch.index[0]) / 1e6  # ms
            all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_time"] = (
                (in_patch[subject_name].cumsum().values * dt).astype(int)  # ms
            )
# Now that we have computed all individual patch and subject values, we iterate again through
# patches and subjects to compute preference scores
for subject_name in subject_names:
    # Get sum of subj cum wheel dists and cum in patch time
    dist_cols = [
        col for col in all_subj_patch_pref.columns 
        if col.startswith(subject_name) and col.endswith("cum_dist")
    ]
    all_cum_dist = np.sum(all_subj_patch_pref[dist_cols].iloc[-1])
    time_cols = [
        col for col in all_subj_patch_pref.columns 
        if col.startswith(subject_name) and col.endswith("cum_time")
    ]
    all_cum_time = np.sum(all_subj_patch_pref[time_cols].iloc[-1])
    for patch in block_patches:
        all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_pref_dist"] = (
            all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_dist"] / all_cum_dist
        )
        all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_pref_time"] = (
            all_subj_patch_pref[subject_name + "_" + patch["patch_name"] + "_" + "cum_time"] / all_cum_time
        )

In [6]:
all_subj_patch_pref

,BAA-1104045_Patch1_cum_dist,BAA-1104045_Patch1_cum_time,BAA-1104045_Patch1_cum_pref_dist,BAA-1104045_Patch1_cum_pref_time,BAA-1104045_Patch2_cum_dist,BAA-1104045_Patch2_cum_time,BAA-1104045_Patch2_cum_pref_dist,BAA-1104045_Patch2_cum_pref_time,BAA-1104045_Patch3_cum_dist,BAA-1104045_Patch3_cum_time,...,BAA-1104047_Patch1_cum_pref_dist,BAA-1104047_Patch1_cum_pref_time,BAA-1104047_Patch2_cum_dist,BAA-1104047_Patch2_cum_time,BAA-1104047_Patch2_cum_pref_dist,BAA-1104047_Patch2_cum_pref_time,BAA-1104047_Patch3_cum_dist,BAA-1104047_Patch3_cum_time,BAA-1104047_Patch3_cum_pref_dist,BAA-1104047_Patch3_cum_pref_time
0,-0.000000,0,-0.000000e+00,0.000000,-0.000000,0,-0.000000e+00,0.000000,-0.000000,0,...,0.000000,0.0000,0.000000,0,0.000000,0.000000,0.000000,0,0.000000,0.000000
1,-0.007670,0,-8.268615e-07,0.000000,-0.003068,0,-3.307446e-07,0.000000,0.000000,0,...,0.000000,0.0000,0.000000,0,0.000000,0.000000,0.000000,0,0.000000,0.000000
2,-0.007670,0,-8.268615e-07,0.000000,0.000000,0,0.000000e+00,0.000000,-0.001534,0,...,0.000000,0.0000,0.000000,0,0.000000,0.000000,0.000000,0,0.000000,0.000000
3,-0.004602,0,-4.961169e-07,0.000000,-0.001534,0,-1.653723e-07,0.000000,0.000000,0,...,0.000000,0.0000,0.000000,0,0.000000,0.000000,0.000000,0,0.000000,0.000000
4,-0.007670,0,-8.268615e-07,0.000000,-0.004602,0,-4.961169e-07,0.000000,-0.001534,0,...,0.000000,0.0000,0.000000,0,0.000000,0.000000,0.000000,0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84896,5.495055,6138,5.923636e-04,0.005295,8349.655653,985941,9.000878e-01,0.850542,921.322143,167112,...,0.039379,0.0505,10778.333242,982080,0.808629,0.826667,2025.933963,145926,0.151993,0.122833
84897,5.502725,6138,5.931904e-04,0.005295,8349.654119,985941,9.000877e-01,0.850542,921.320609,167112,...,0.039379,0.0505,10778.333242,982080,0.808629,0.826667,2025.933963,145926,0.151993,0.122833
84898,5.502725,6138,5.931904e-04,0.005295,8349.654119,985941,9.000877e-01,0.850542,921.320609,167112,...,0.039379,0.0505,10778.327106,982080,0.808629,0.826667,2025.929361,145926,0.151993,0.122833
84899,5.502725,6138,5.931904e-04,0.005295,8349.658721,985941,9.000882e-01,0.850542,921.326745,167112,...,0.039379,0.0505,10778.327106,982080,0.808629,0.826667,2025.929361,145926,0.151993,0.122833


---